In [ ]:
# Cell 1: Install Dependencies
!pip install -q langchain langchain-community langchain-groq langgraph chromadb sentence-transformers pymupdf

In [4]:
# Cell 2: Imports
import fitz  # pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional
import warnings
warnings.filterwarnings("ignore")

print("All imports successful")

All imports successful


In [5]:
# Cell 3: Configuration
import os

GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

PDF_PATH = "/content/Knowledge Base.pdf"

print("Config set")

Config set


In [6]:
# Cell 4: Load & Chunk PDF with Accuracy-Optimized Settings
def load_pdf(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

raw_text = load_pdf(PDF_PATH)

# Small chunks with high overlap = better accuracy for FAQ-style docs
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100,
    separators=["\n\n", "\n", "Q:", "?"]
)

chunks = splitter.create_documents([raw_text])
print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content}")

Total chunks created: 6

Sample chunk:
TechCorp Customer Support Knowledge Base 
Q: How do I reset my password? Go to login page → click "Forgot Password" → enter your 
email → check inbox for reset link. Link expires in 30 minutes. 
Q: How do I cancel my subscription? Login → Settings → Billing → Cancel Subscription.


In [7]:
# Cell 5: Build Vector Store with Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="techcorp_support"
)

retriever = vectorstore.as_retriever(
    search_type="mmr",  # MMR reduces redundancy, improves accuracy
    search_kwargs={"k": 3, "fetch_k": 6}
)

print("Vector store ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store ready


In [8]:
# Cell 6: Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,  # 0 = deterministic = more accurate
    api_key=GROQ_API_KEY
)

print("LLM ready")

LLM ready


In [9]:
# Cell 7: Define Graph State & Prompt
from langchain_core.prompts import ChatPromptTemplate

class GraphState(TypedDict):
    query: str
    context: str
    answer: str
    confidence: str  # "high" or "low"
    escalated: bool

PROMPT = ChatPromptTemplate.from_template("""
You are TechCorp's customer support assistant.
Answer in a warm, helpful tone like a real support agent.
Use the context below only. Keep it conversational but accurate.
If the answer is not in the context, reply exactly: "I don't know"

Context:
{context}

Question: {query}

Answer:""")

print("State and prompt ready")

State and prompt ready


In [10]:
# Cell 8: Define Graph Nodes
def retrieve_node(state: GraphState) -> GraphState:
    docs = retriever.invoke(state["query"])
    state["context"] = "\n".join([d.page_content for d in docs])
    return state

def generate_node(state: GraphState) -> GraphState:
    chain = PROMPT | llm
    response = chain.invoke({"query": state["query"], "context": state["context"]})
    answer = response.content.strip()
    state["answer"] = answer
    state["confidence"] = "low" if "i don't know" in answer.lower() else "high"
    return state

def hitl_node(state: GraphState) -> GraphState:
    print(f"\n⚠️ HITL ESCALATION")
    print(f"Query: {state['query']}")
    human_response = input("Human Agent Answer: ")
    state["answer"] = human_response
    state["escalated"] = True
    return state

def route(state: GraphState) -> str:
    return "hitl" if state["confidence"] == "low" else END

print("Nodes ready")

Nodes ready


In [11]:
# Cell 9: Build & Compile LangGraph
graph = StateGraph(GraphState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_node("hitl", hitl_node)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_conditional_edges("generate", route, {"hitl": "hitl", END: END})
graph.add_edge("hitl", END)

app = graph.compile()
print("Graph compiled successfully")

Graph compiled successfully


In [13]:
# Cell 10: Gradio Chat UI
!pip install -q gradio

import gradio as gr

def chat(query, history):
    result = app.invoke({
        "query": query,
        "context": "",
        "answer": "",
        "confidence": "",
        "escalated": False
    })
    answer = result["answer"]
    if result["escalated"]:
        answer += "\n\n✅ *Resolved via Human Agent*"
    return answer

ui = gr.ChatInterface(
    fn=chat,
    title="Customer Support Assistant",
    description="Ask anything about your account, billing, or technical issues.",
    examples=["How do I reset my password?", "What is the refund policy?", "How do I contact support?"],
    theme=gr.themes.Soft()
)

ui.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://465be82088002526c4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
